In [ ]:
# run if necessary
#%pip install numpy scipy matplotlib
#%pip install torch torchvision

# this file is mostly junk/scratchwork ish

In [200]:
import numpy as np

In [189]:
rows,cols = 5,7
B = np.random.choice([0,1], size=(rows,cols), p=[0.6,0.4])

In [ ]:
def normalized_biadjacency(B):
    B = B.astype(float)
    deg_row = B.sum(axis=1)
    deg_col = B.sum(axis=0)

    deg_row[deg_row == 0] = 1
    deg_col[deg_col == 0] = 1

    row_scaling = 1 / np.sqrt(deg_row).reshape(-1, 1)  # shape: (rows, 1)
    col_scaling = 1 / np.sqrt(deg_col).reshape(1, -1)  # shape: (1, cols)

    B_tilde = B * row_scaling * col_scaling
    
    return(B_tilde)

def build_block_matrix(B):
    m,n = B.shape
    A = np.block([[np.zeros((m,m)),B],
                   [B.T, np.zeros((n,n))]])
    return A

def filter(x): #replace this one with anything
    return np.exp(-x)
def h(x): #even part
    return 0.5*(filter(x) + filter(-x))
def g(x): #odd part
    return 0.5*(filter(x) - filter(-x))


In [191]:
B_tilde = normalized_biadjacency(B)

In [192]:
B_tilde

array([[0.        , 0.25819889, 0.31622777, 0.        , 0.31622777,
        0.25819889, 0.4472136 ],
       [0.28867513, 0.        , 0.35355339, 0.25      , 0.        ,
        0.28867513, 0.        ],
       [0.        , 0.        , 0.        , 0.5       , 0.        ,
        0.        , 0.        ],
       [0.25819889, 0.25819889, 0.        , 0.2236068 , 0.31622777,
        0.25819889, 0.        ],
       [0.33333333, 0.33333333, 0.        , 0.28867513, 0.        ,
        0.        , 0.        ]])

In [193]:
A = build_block_matrix(B_tilde)
print(A)

[[0.         0.         0.         0.         0.         0.
  0.25819889 0.31622777 0.         0.31622777 0.25819889 0.4472136 ]
 [0.         0.         0.         0.         0.         0.28867513
  0.         0.35355339 0.25       0.         0.28867513 0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.5        0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.25819889
  0.25819889 0.         0.2236068  0.31622777 0.25819889 0.        ]
 [0.         0.         0.         0.         0.         0.33333333
  0.33333333 0.         0.28867513 0.         0.         0.        ]
 [0.         0.28867513 0.         0.25819889 0.33333333 0.
  0.         0.         0.         0.         0.         0.        ]
 [0.25819889 0.         0.         0.25819889 0.33333333 0.
  0.         0.         0.         0.         0.         0.        ]
 [0.31622777 0.35355339 0.         0.         0.         0.
  0.         

In [194]:
eigs, vects = np.linalg.eig(A)
v = np.diag(eigs)
u = vects
u_inv = np.linalg.inv(vects)

In [195]:
eigs = filter(eigs)
v = np.diag(eigs)
u @ v @ u_inv

array([[ 1.28299578e+00,  1.02996808e-01,  2.71184285e-03,
         1.28080506e-01,  4.99207240e-02, -2.53871039e-02,
        -2.97846534e-01, -3.57029466e-01, -2.22531092e-02,
        -3.58373960e-01, -3.02138067e-01, -4.88414744e-01],
       [ 1.02996808e-01,  1.18766067e+00,  6.77788417e-02,
         1.12905304e-01,  9.25430686e-02, -3.25580075e-01,
        -2.77809197e-02, -3.85600499e-01, -2.92900420e-01,
        -2.18918291e-02, -3.24194925e-01, -1.47590940e-02],
       [ 2.71184285e-03,  6.77788417e-02,  1.12975100e+00,
         6.13668490e-02,  7.76464734e-02, -1.97878189e-02,
        -1.36097183e-02, -7.90462219e-03, -5.38442108e-01,
        -6.40394777e-03, -1.15449003e-02, -2.39019871e-04],
       [ 1.28080506e-01,  1.12905304e-01,  6.13668490e-02,
         1.18737345e+00,  1.28062797e-01, -2.98142201e-01,
        -2.98309469e-01, -2.58203026e-02, -2.68057018e-01,
        -3.48462321e-01, -2.94975753e-01, -1.84025698e-02],
       [ 4.99207240e-02,  9.25430686e-02,  7.7646473

In [204]:
U,S,Vh = np.linalg.svd(B_tilde, full_matrices=True)
Sigma = np.zeros(B_tilde.shape)
print(S)
S = g(S)
np.fill_diagonal(Sigma,S)
U@Sigma@Vh


[1.         0.65213438 0.41879842 0.35700676 0.25481437]


array([[-2.53871039e-02, -2.97846534e-01, -3.57029466e-01,
        -2.22531092e-02, -3.58373960e-01, -3.02138067e-01,
        -4.88414744e-01],
       [-3.25580075e-01, -2.77809197e-02, -3.85600499e-01,
        -2.92900420e-01, -2.18918291e-02, -3.24194925e-01,
        -1.47590940e-02],
       [-1.97878189e-02, -1.36097183e-02, -7.90462219e-03,
        -5.38442108e-01, -6.40394777e-03, -1.15449003e-02,
        -2.39019871e-04],
       [-2.98142201e-01, -2.98309469e-01, -2.58203026e-02,
        -2.68057018e-01, -3.48462321e-01, -2.94975753e-01,
        -1.84025698e-02],
       [-3.70154667e-01, -3.65630886e-01, -1.54739914e-02,
        -3.33134720e-01, -1.80502148e-02, -2.33170600e-02,
        -7.02405439e-03]])

In [198]:
filtered_A = np.block([[]])

ValueError: List at arrays[0] cannot be empty